# TechCorp AI — Fine-tuning LoRA Médical

**Modèle :** `microsoft/Phi-3.5-mini-instruct`  
**Dataset :** `ruslanmv/ai-medical-chatbot` (256 916 conversations)  
**Méthode :** QLoRA 4-bit (r=16, alpha=32)  
**GPU requis :** T4 minimum (Colab Pro recommandé pour A100)

> Ce modèle est expérimental — ne pas déployer en production médicale sans validation.

In [ ]:
# Vérification GPU
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Installation dépendances
!pip install -q transformers==4.44.0 peft==0.12.0 bitsandbytes==0.43.3 accelerate==0.34.0 trl==0.10.1 datasets

In [ ]:
# Téléchargement dataset médical
from datasets import load_dataset
import json

print('Chargement ruslanmv/ai-medical-chatbot...')
ds_raw = load_dataset('ruslanmv/ai-medical-chatbot', split='train')
print(f'Dataset chargé : {len(ds_raw):,} entrées')
print('Colonnes :', ds_raw.column_names)
print('Exemple :', ds_raw[0])

In [ ]:
# Nettoyage et formatage
SYSTEM_PROMPT = (
    'You are a medical AI assistant. A patient is asking a health question. '
    'Provide a helpful, accurate, and empathetic response. '
    'Always recommend consulting a qualified healthcare professional for diagnosis and treatment.'
)
MAX_SAMPLES = 10000  # Augmenter si GPU A100

def is_valid(entry):
    patient = entry.get('Patient', '').strip()
    doctor = entry.get('Doctor', '').strip()
    return 20 <= len(patient) <= 2000 and 30 <= len(doctor) <= 3000

def format_entry(entry):
    return (
        f'<|system|>\n{SYSTEM_PROMPT}<|end|>\n'
        f'<|user|>\n{entry["Patient"].strip()}<|end|>\n'
        f'<|assistant|>\n{entry["Doctor"].strip()}<|end|>'
    )

valid = [e for e in ds_raw if is_valid(e)]
subset = valid[:MAX_SAMPLES]
formatted = [{'text': format_entry(e)} for e in subset]

print(f'Entrées valides : {len(valid):,}')
print(f'Sous-ensemble utilisé : {len(formatted):,}')
print('\nExemple formaté :')
print(formatted[0]['text'][:500])

In [ ]:
# Conversion en Dataset HuggingFace
from datasets import Dataset

dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds = split['test']

print(f'Train : {len(train_ds)} | Eval : {len(eval_ds)}')

In [ ]:
# Chargement modèle QLoRA 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False

print('Modèle chargé')

In [ ]:
# Configuration LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f'Paramètres entraînables : {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

In [ ]:
# Entraînement
import math
from transformers import TrainingArguments
from trl import SFTTrainer

EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
OUTPUT_DIR = '/content/phi35_medical_lora'

steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
eval_steps = max(50, steps_per_epoch // 2)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=eval_steps,
    save_strategy='epoch',
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
)

print('Démarrage entraînement...')
result = trainer.train()
print(f'Loss finale : {result.training_loss:.4f}')
print(f'Temps total : {result.metrics["train_runtime"]:.0f}s')

In [ ]:
# Sauvegarde et test
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modèle sauvegardé : {OUTPUT_DIR}')

# Test rapide
from peft import PeftModel

test_prompt = '<|system|>\nYou are a medical AI assistant.<|end|>\n<|user|>\nWhat is hypertension?<|end|>\n<|assistant|>\n'
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('\nTest réponse médicale :')
print(response)

In [ ]:
# Métriques finales
metrics = trainer.evaluate()
perplexity = math.exp(metrics['eval_loss'])

print('=== MÉTRIQUES FINALES ===')
print(f'Eval Loss     : {metrics["eval_loss"]:.4f}')
print(f'Perplexité    : {perplexity:.2f}')
print(f'Samples train : {len(train_ds):,}')
print(f'Epochs        : {EPOCHS}')
print(f'LoRA r        : 16 | alpha : 32')
print(f'Modèle        : {OUTPUT_DIR}')